# Notebook 4: The Attention Bottleneck
## Expressibility Tests, Honest Partitioning, and the Hybrid Path Forward

This notebook presents the analytical conclusion: **softmax-normalized attention is fundamentally analog-hostile**, and the honest engineering response is a hybrid digital/analog partition (Option B).


# Setup

This notebook is self-contained. The cell below ensures the `neuro-analog` repository is on the path and that `numpy` and `matplotlib` are available. No GPU-heavy dependencies are required.


In [ ]:
"""Lightweight bootstrap: clone repo if missing, ensure numpy/matplotlib."""
import sys, os, subprocess, importlib

repo_name = "neuro-analog"
repo_path = None
cwd = os.getcwd()
parts = cwd.replace("\\\\", "/").split("/")
for i in range(len(parts), 0, -1):
    candidate = "/".join(parts[:i])
    if candidate.endswith(repo_name) and os.path.isdir(os.path.join(candidate, "neuro_analog")):
        repo_path = candidate
        break
if repo_path is None:
    colab_default = f"/content/{repo_name}"
    if os.path.isdir(os.path.join(colab_default, "neuro_analog")):
        repo_path = colab_default
if repo_path is None:
    print("Cloning neuro-analog...")
    clone_target = f"/content/{repo_name}"
    subprocess.run(
        ["git", "clone", "https://github.com/apumutyala/neuro-analog.git", clone_target],
        check=True, capture_output=True, text=True
    )
    repo_path = clone_target
    print(f"Cloned to {repo_path}")
else:
    print(f"Found repo at: {repo_path}")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

for pkg in ["numpy", "matplotlib"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run(["pip", "install", "-q", pkg], check=True, capture_output=True, text=True)

import numpy as np
import matplotlib.pyplot as plt
print("Ready.")


# Notebook 4: The Attention Bottleneck
## Expressibility Tests, Honest Partitioning, and the Hybrid Path Forward

This notebook is **pre-run and static** (no live execution needed). It presents the analytical conclusion: **softmax-normalized attention is fundamentally analog-hostile**, and the honest engineering response is a hybrid digital/analog partition (Option B).

**Key thesis:** Transformers win on scale because attention is a *global read* — every token attends to every other token. But an analog crossbar computes matrix-vector multiply in O(1) physical time. The problem isn't the MVM; it's the **global normalization** in softmax: `exp(q·k) / sum_j exp(q·k_j)` requires a sum over all tokens, which is a collective operation across the entire array. Analog hardware has no efficient way to compute a global sum without an ADC round-trip or a dedicated digital accumulator.


## 3.1 Why Attention Fails in Analog: The Normalization Wall

**The operation:**
```
Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) @ V
```

**What maps to analog easily:**
- `Q = X @ W_q` — MVM on crossbar ✓
- `K = X @ W_k` — MVM on crossbar ✓
- `V = X @ W_v` — MVM on crossbar ✓
- `O = Attn @ W_o` — MVM on crossbar ✓
- `S = Q @ K^T` — MVM on crossbar ✓ (score matrix)

**What does NOT map to analog:**
- `softmax(S) = exp(S_ij) / sum_j exp(S_ij)` — **global row-wise normalization**

**Why it's hard:**
1. `exp()` of an analog voltage requires a transistor in subthreshold — possible but noisy and temperature-sensitive.
2. The **denominator `sum_j`** requires summing over an entire row of the score matrix. In a crossbar, this is a *column-wise* current summation, but softmax needs a *row-wise* sum. You'd need to transpose the array or add a dedicated digital accumulator.
3. **Division** (`numerator / denominator`) is not a native analog operation. You need a Gilbert cell analog divider (power-hungry, low precision) or an ADC → digital division → DAC round-trip.
4. Even if you build it, the **accuracy requirement is extreme**: a 4096-token context means the denominator sums 4096 exponentials. A 1% error in any `exp(S_ij)` propagates multiplicatively through the normalized attention weight. Analog precision (~5-8 bits effective) is insufficient.

**Conclusion:** The bottleneck is not the MVMs — it's the **global normalization**. This is a fundamental expressibility gap, not an engineering detail.


In [ ]:
# VISUAL: Numerical demonstration of softmax sensitivity
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
d_k = 64
seq_len = 128

# Simulate a QK^T row
scores = np.random.randn(seq_len) / np.sqrt(d_k)

# Add analog-like multiplicative noise (5% sigma = typical RRAM mismatch)
noisy_scores = scores * (1 + 0.05 * np.random.randn(seq_len))

# True and noisy softmax
def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

p_true = softmax(scores)
p_noisy = softmax(noisy_scores)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].stem(p_true, linefmt='C0-', markerfmt='C0o', basefmt=' ', label='Nominal')
axes[0].set_title("Nominal Softmax Distribution")
axes[0].set_xlabel("Token index j")
axes[0].set_ylabel("Attention weight")
axes[1].stem(p_noisy, linefmt='C1-', markerfmt='C1o', basefmt=' ', label='With 5% mismatch')
axes[1].set_title("Softmax with 5% Analog Mismatch")
axes[1].set_xlabel("Token index j")
axes[1].set_ylabel("Attention weight")
plt.suptitle("Softmax is Fragile: 5% Mismatch Completely Reshapes Attention Weights")
plt.tight_layout()
plt.show()

# Quantify the damage
kl_div = np.sum(p_true * np.log(p_true / (p_noisy + 1e-10)))
max_shift = np.max(np.abs(p_true - p_noisy))
print(f"KL divergence: {kl_div:.4f}")
print(f"Max attention weight shift: {max_shift:.4f}")
print(f"Top-1 token changed: {np.argmax(p_true) != np.argmax(p_noisy)}")

## 3.2 Four Options for Attention in Analog AI

| Option | Name | Description | Feasibility | Trade-off |
|--------|------|-------------|-------------|-----------|
| **A** | Full Analog Attention | Implement softmax, exp, division, and global sum entirely in analog | ❌ **Impossible** | Precision catastrophe; no known analog divider achieves >6-bit accuracy for dynamic-range softmax inputs |
| **B** | **Hybrid (Recommended)** | Analog MVMs for Q, K, V, O; digital softmax in a small DSP/CPU; explicit A/D boundaries | ✅ **Mature path** | Adds ADC/DAC latency (~μs per token) and power, but preserves analog speedup where it matters (the O(n²) MVMs) |
| **C** | Analog Approximation | Replace softmax with analog-friendly alternatives: WTA (winner-take-all), L2-normalized dot product, or sigmoid-mixture attention | ⚠️ **Research** | May preserve 80-90% of transformer performance; no proven analog circuit for any of these at >1000-token scale |
| **D** | Attention-Free Architecture | Replace transformer with Mamba, S4, or other state-space models that are natively analog-compatible | ✅ **Long-term** | Changes the algorithm, not the hardware. SSMs are linear recurrences = analog-native. But retraining and ecosystem migration cost is massive. |



In [ ]:
# VISUAL: Hybrid Option B schematic
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 5))

# Analog blocks (green)
analog_blocks = [
    (0.5, 3.5, 1.5, 0.8, "X @ W_q\n(Analog MVM)"),
    (2.5, 3.5, 1.5, 0.8, "X @ W_k\n(Analog MVM)"),
    (4.5, 3.5, 1.5, 0.8, "X @ W_v\n(Analog MVM)"),
    (2.5, 1.5, 2.0, 0.8, "S = Q @ K^T\n(Analog MVM)"),
    (6.5, 1.5, 1.5, 0.8, "Attn @ V\n(Analog MVM)"),
    (9.0, 1.5, 1.5, 0.8, "O @ W_o\n(Analog MVM)"),
]

for x, y, w, h, label in analog_blocks:
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05",
                                    facecolor='lightgreen', edgecolor='darkgreen', lw=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=8, wrap=True)

# Digital block (blue)
rect = mpatches.FancyBboxPatch((4.5, 2.5), 2.0, 0.6, boxstyle="round,pad=0.05",
                                facecolor='lightblue', edgecolor='darkblue', lw=2)
ax.add_patch(rect)
ax.text(5.5, 2.8, "softmax(S)\n(Digital DSP)", ha='center', va='center', fontsize=8, fontweight='bold')

# ADC/DAC arrows
ax.annotate("ADC", xy=(5.5, 3.1), xytext=(5.5, 3.5),
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
            fontsize=7, ha='center', color='red')
ax.annotate("DAC", xy=(5.5, 2.5), xytext=(5.5, 2.1),
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
            fontsize=7, ha='center', color='red')

# Connections
ax.plot([1.25, 2.5], [3.5, 3.9], 'k-', lw=1)
ax.plot([3.25, 3.5], [3.5, 2.3], 'k-', lw=1)
ax.plot([5.25, 5.5], [3.5, 3.1], 'k-', lw=1)
ax.plot([5.5, 5.5], [2.1, 1.9], 'k-', lw=1)
ax.plot([6.5, 7.25], [1.9, 1.9], 'k-', lw=1)
ax.plot([8.0, 9.0], [1.9, 1.9], 'k-', lw=1)

# Token stream labels
ax.text(0.3, 4.5, "Token stream X\n(sequentially processed)", fontsize=9, style='italic')
ax.text(10.5, 1.9, "Output\n(analog)", fontsize=9, ha='center')

ax.set_xlim(0, 11)
ax.set_ylim(0.5, 5)
ax.axis('off')
ax.set_title("Option B: Hybrid Attention Partition\nAnalog MVMs + Digital Softmax + Explicit A/D Boundaries", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3.3 SSM as the Analog-Native Alternative (Option D Preview)

**Why SSMs are analog-friendly:**
- Linear recurrence: `h[t] = A @ h[t-1] + B @ u[t]`
- `A` is diagonal (S4D): element-wise multiply = analog RC decay circuit
- `B @ u` and `C @ h` are MVMs on crossbars
- No global normalization, no softmax, no attention weights
- The entire layer is a composition of MVMs and element-wise ops — both analog-native

**Why this is the long-term bet:**
- If analog in-memory compute achieves 1000× speedup on MVMs, the winning architecture is the one that replaces attention with more MVMs.
- SSMs (Mamba, S4, RWKV) are exactly this: they trade the O(n²) attention MVM for O(n) state updates that are *also* MVMs, just smaller ones repeated over time.
- The hardware-software co-design thesis: *don't force analog to emulate digital transformers; design algorithms that leverage what analog does well.*

**Notebook 1 already demonstrated this:** The SSM fallback in `LinearSSMCkt` is a plain `BaseAnalogCkt` that runs a diagonal state-space recurrence. It compiled and solved without any attention mechanism.


In [ ]:
# VISUAL: Attention O(n²) vs SSM O(n) complexity, with analog speedup annotations
import numpy as np
import matplotlib.pyplot as plt

seq_lens = np.array([64, 128, 256, 512, 1024, 2048, 4096])

# Digital baseline (arbitrary units)
attn_digital = seq_lens ** 2
ssm_digital = seq_lens * 64  # d_state=64, linear in sequence length

# Analog speedup: MVMs get 100x, but attention still has O(n²) MVMs
attn_analog = seq_lens ** 2 / 100
ssm_analog = seq_lens * 64 / 100

# Hybrid attention: analog MVMs + digital softmax overhead (fixed per token, ~1000 ops)
hybrid_analog = (seq_lens ** 2 / 100) + (seq_lens * 1000)

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(seq_lens, attn_digital, 'o--', color='gray', label='Attention (digital)', alpha=0.5)
ax.loglog(seq_lens, attn_analog, 'o-', color='red', label='Attention (all analog) — infeasible', lw=2)
ax.loglog(seq_lens, hybrid_analog, 's-', color='blue', label='Hybrid Option B (analog MVM + digital softmax)', lw=2)
ax.loglog(seq_lens, ssm_analog, '^-', color='green', label='SSM (analog) — Option D', lw=2)

ax.set_xlabel("Sequence Length")
ax.set_ylabel("Relative Compute Cost (log scale)")
ax.set_title("Compute Scaling: Attention vs SSM Under Analog Speedup")
ax.legend(loc='upper left')
ax.grid(True, which='both', ls='--', alpha=0.3)

# Annotate the crossover
ax.annotate("Crossover:\nSSM wins at all scales", xy=(2048, ssm_analog[-2]),
            xytext=(512, ssm_analog[-2] * 3),
            arrowprops=dict(arrowstyle='->', color='green', lw=1.5),
            fontsize=9, color='green')

plt.tight_layout()
plt.show()

## 3.4 Summary

**Can you analogize a transformer?**

> The MVMs — yes. The softmax — no. The bottleneck isn't compute; it's the global normalization. Analog hardware has no efficient mechanism for row-wise division by a dynamically-varying sum across thousands of tokens.

**The recommended path (Option B):**
- Analog islands for Q, K, V, O MVMs (the bulk of FLOPs)
- Digital bridge for softmax (small, fixed overhead)
- Explicit A/D boundaries with latency/power modeled
- No change to training, inference API, or model weights

**The research bet (Option D):**
- SSMs replace attention with linear recurrences
- Entirely analog-native: no global ops, no normalization
- Mamba is already competitive with transformers on many tasks
- Hardware-software co-design: *build algorithms that leverage analog's strengths, don't force analog to emulate digital*

**Key insight:**
> The question isn't whether analog can run a transformer. The question is whether the architecture you'd design if you started from analog physics looks like a transformer. The answer is no — it looks like a state-space model.
